# Table IV rerun on the same seeds as Table V, plus the stack boundary-probability count

Closes the two remaining numeric gaps in the paper:

1. **Table IV (random split / group-aware / group-aware + recoded bedtime)** is recomputed with *exactly* the pipeline of the Table V notebook (same seeds 0-4, same `StratifiedGroupKFold(random_state=seed)` folds, same Random Forest settings, 500 trees). The Random Forest group-aware ROC-AUC and PR-AUC should therefore equal Table V (0.682 / 0.519). The cell prints that check.
2. **The "11,868 inner OOF probabilities" sentence** in Section III-D is recounted: total inner out-of-fold probabilities and the share of Extra Trees outputs that are exactly 0 and exactly 1, for seeds 0-2 (old stack run) and seeds 0-4 (current run).
3. Optional: the 50-permutation label test and the paired bootstrap differences quoted in Section IV-F.

Run top to bottom with `FAST = False`. Only the Kaggle CSV is needed. Nothing is typed in by hand: every number comes from the data.

In [ ]:
import numpy as np, pandas as pd, re, os, shutil, pathlib, warnings
warnings.filterwarnings("ignore")
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              GradientBoostingClassifier, HistGradientBoostingClassifier)
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score
try:
    from xgboost import XGBClassifier
except ImportError:
    raise SystemExit("Run  !pip install xgboost  and run again.")

DATA_FILE = None        # None = Colab upload button; or a path such as "igd_data.csv"
FAST = False            # True = quick trial only, never for paper numbers
SEEDS   = [0] if FAST else [0, 1, 2, 3, 4]
N_TREES = 40 if FAST else 500
N_BOOT  = 50 if FAST else 1000
N_PERM  = 3 if FAST else 50
RUN_PERM = True         # label-permutation test (slowest cell; set False to skip)
N_SPLITS = 5
OUT = "table4_outputs"; os.makedirs(OUT, exist_ok=True)
print("seeds", SEEDS, "| trees", N_TREES, "| bootstrap", N_BOOT, "| permutations", N_PERM)

## 1. Load data, label, groups

In [ ]:
if DATA_FILE is None:
    try:
        from google.colab import files
        up = files.upload(); DATA_FILE = list(up)[0]
    except ImportError:
        raise SystemExit("Set DATA_FILE to the CSV path and run again.")
df = pd.read_csv(DATA_FILE)
cg = [c for c in df.columns if "cgpa" in c.lower()]; assert len(cg) == 1, cg
y = (df[cg[0]] < 3.00).astype(int).to_numpy()
X = df.drop(columns=[cg[0]]).copy()
X = X.drop(columns=[c for c in X.columns if c.lower().startswith("unnamed")], errors="ignore")
groups = X.astype(str).agg("|".join, axis=1).factorize()[0]
print("rows", len(y), "| predictors", X.shape[1], "(paper 19) | at-risk", round(y.mean(), 3),
      "(paper ~0.297) | groups", len(np.unique(groups)), "(paper 605)")
assert X.shape[1] == 19, list(X.columns)
if len(np.unique(groups)) != 605:
    print("WARNING: group count differs from the paper - check the file before trusting results.")

## 2. Bedtime recoding (hours elapsed after 18:00)

The raw answers are parsed to a clock hour and recoded as `(hour - 18) mod 24`. **Read the printed mapping.** If any answer cannot be parsed, or the mapping looks wrong, put the correct clock hours (0-24) in `BEDTIME_MAP` and re-run this cell.

In [ ]:
bed = [c for c in X.columns if re.search(r"go\s*to\s*sleep", c, flags=re.I)]
assert len(bed) == 1, f"Bedtime column not found uniquely: {bed}. Set it by hand."
bed = bed[0]
BEDTIME_MAP = {}     # e.g. {"10 PM - 12 AM": 23.0, "After 2 AM": 3.0}; overrides the parser

def parse_clock(s):
    m = re.search(r"(\d{1,2})(?::(\d{2}))?\s*(a\.?m\.?|p\.?m\.?)?", str(s).lower())
    if not m: return None
    h, mi, ap = int(m.group(1)), int(m.group(2) or 0), m.group(3)
    if ap:
        if ap[0] == "p" and h < 12: h += 12
        if ap[0] == "a" and h == 12: h = 0
    return (h % 24) + mi / 60

vals = sorted(X[bed].astype(str).unique())
cmap = {v: BEDTIME_MAP.get(v, parse_clock(v)) for v in vals}
for v in vals: print(f"{v!r:40} -> clock hour {cmap[v]}")
assert all(h is not None for h in cmap.values()), "Unparsed answers: fill BEDTIME_MAP and re-run."
X_rec = X.copy()
X_rec[bed] = X[bed].astype(str).map(lambda v: (cmap[v] - 18) % 24)

## 3. Models and helpers (identical settings to the Table V notebook)

In [ ]:
MODEL_NAMES = ["Random Forest", "Extra Trees", "Hist. Gradient Boosting", "XGBoost",
               "Gradient Boosting", "Logistic Regression", "Dummy (prior)"]
def make_models(seed):
    return {
      "Random Forest": RandomForestClassifier(n_estimators=N_TREES, max_features="sqrt", random_state=seed, n_jobs=-1),
      "Extra Trees":   ExtraTreesClassifier(n_estimators=N_TREES, max_features="sqrt", random_state=seed, n_jobs=-1),
      "Hist. Gradient Boosting": HistGradientBoostingClassifier(max_iter=100, learning_rate=0.1, min_samples_leaf=20, random_state=seed),
      "XGBoost": XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, subsample=0.8, colsample_bytree=0.8,
                               random_state=seed, n_jobs=-1, eval_metric="logloss", verbosity=0),
      "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=seed),
      "Logistic Regression": LogisticRegression(penalty="l2", C=1.0, max_iter=2000),
      "Dummy (prior)": DummyClassifier(strategy="prior"),
    }
def make_pre(Xd):
    cats = [c for c in Xd.columns if not pd.api.types.is_numeric_dtype(Xd[c])]
    return ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cats)],
                             remainder="passthrough", sparse_threshold=0.0)
def fit_pred(model, Xa, ya, Xb):
    return Pipeline([("pre", make_pre(Xa)), ("m", clone(model))]).fit(Xa, ya).predict_proba(Xb)[:, 1]
def group_splits(yv, gv, k, seed):
    return list(StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed).split(np.zeros(len(yv)), yv, gv))

## 4. Table IV: random split, group-aware, group-aware + recoded bedtime

In [ ]:
SCHEMES = {"random": (X, False), "group": (X, True), "group_recoded": (X_rec, True)}
rows, store = [], {m: [] for m in MODEL_NAMES}     # store = group-aware held-out predictions for the bootstrap
for scheme, (Xd, use_g) in SCHEMES.items():
    for seed in SEEDS:
        if use_g: splits = group_splits(y, groups, N_SPLITS, seed)
        else:     splits = list(StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed).split(np.zeros(len(y)), y))
        for fold, (tr, va) in enumerate(splits):
            if use_g: assert not set(groups[tr]) & set(groups[va])
            Xtr, ytr = Xd.iloc[tr].reset_index(drop=True), y[tr]
            Xva, yva = Xd.iloc[va].reset_index(drop=True), y[va]
            for name, mdl in make_models(seed).items():
                p = fit_pred(mdl, Xtr, ytr, Xva)
                rows.append(dict(scheme=scheme, seed=seed, fold=fold, model=name,
                                 roc_auc=roc_auc_score(yva, p), pr_auc=average_precision_score(yva, p)))
                if scheme == "group": store[name].append(dict(y=yva, gi=groups[va], p=p))
        print(scheme, "seed", seed, "done", flush=True)
res = pd.DataFrame(rows); res.to_csv(f"{OUT}/table4_fold_level.csv", index=False)

roc = res.pivot_table(index="model", columns="scheme", values="roc_auc", aggfunc="mean").loc[MODEL_NAMES]
pr  = res[res.scheme == "group"].groupby("model")["pr_auc"].mean().loc[MODEL_NAMES]
t4 = pd.DataFrame({"Random split": roc["random"], "Group": roc["group"], "Group + recoded": roc["group_recoded"],
                   "Delta": roc["group"] - roc["random"], "Group PR-AUC": pr})
print(t4.round(3).to_string())
rf = res[(res.scheme == "group") & (res.model == "Random Forest")]
print("\nCHECK vs Table V (seed-for-seed identical pipeline): RF group-aware ROC-AUC %.3f, PR-AUC %.3f  (Table V: 0.682 / 0.519 with 5 seeds, 500 trees)"
      % (rf.roc_auc.mean(), rf.pr_auc.mean()))

## 5. Group-level bootstrap CIs and paired differences

One resample of whole answer-pattern groups per replicate, applied to every seed, fold and model (so paired differences share the same resamples). Each held-out fold is scored with those group counts as sample weights.

In [ ]:
G = len(np.unique(groups)); rng = np.random.default_rng(0)
boot = {m: np.full(N_BOOT, np.nan) for m in MODEL_NAMES}
for b in range(N_BOOT):
    counts = np.bincount(rng.integers(0, G, G), minlength=G)
    for m in MODEL_NAMES:
        v = []
        for u in store[m]:
            w = counts[u["gi"]]
            if w[u["y"] == 1].sum() == 0 or w[u["y"] == 0].sum() == 0: continue
            v.append(roc_auc_score(u["y"], u["p"], sample_weight=w))
        boot[m][b] = np.mean(v)
ci = pd.DataFrame({m: np.nanpercentile(boot[m], [2.5, 97.5]) for m in MODEL_NAMES}, index=["lo", "hi"]).T
print(ci.round(3).to_string())
PAIRS = [("Random Forest", "Extra Trees"), ("Random Forest", "XGBoost"), ("Extra Trees", "XGBoost"),
         ("Random Forest", "Hist. Gradient Boosting")]
pair_rows = []
for a, c in PAIRS:
    d = boot[a] - boot[c]; lo, hi = np.nanpercentile(d, [2.5, 97.5])
    pair_rows.append(dict(a=a, b=c, diff=t4.loc[a, "Group"] - t4.loc[c, "Group"], lo=lo, hi=hi))
pairs = pd.DataFrame(pair_rows); print(); print(pairs.round(3).to_string(index=False))

## 6. Label-permutation null test (primary model: Random Forest)

In [ ]:
perm_null, observed = None, None
if RUN_PERM:
    seed = SEEDS[0]; prng = np.random.default_rng(12345); null = []
    for k in range(N_PERM):
        yp = prng.permutation(y); aucs = []
        for tr, va in group_splits(yp, groups, N_SPLITS, seed):
            p = fit_pred(make_models(seed)["Random Forest"], X.iloc[tr].reset_index(drop=True), yp[tr], X.iloc[va].reset_index(drop=True))
            if len(np.unique(yp[va])) == 2: aucs.append(roc_auc_score(yp[va], p))
        null.append(np.mean(aucs))
    null = np.array(null)
    observed = res[(res.scheme == "group") & (res.model == "Random Forest") & (res.seed == seed)].roc_auc.mean()
    p_emp = (1 + (null >= observed).sum()) / (N_PERM + 1)
    print(f"null range {null.min():.3f}-{null.max():.3f}, mean {null.mean():.3f}; observed (seed {seed}) {observed:.3f}; empirical p = {p_emp:.4f} (= 1/{N_PERM+1} if no null value reaches it)")
    pd.DataFrame({"null_roc_auc": null}).to_csv(f"{OUT}/permutation_null.csv", index=False)

## 7. Recount of the Section III-D sentence (inner OOF probabilities, Extra Trees 0 / 1)

Uses the same inner splits as the Table V notebook (`seed + 1000 + fold`, 5 inner folds, same Extra Trees settings). The count of inner OOF probabilities per base model is the sum of the training-set sizes over all outer folds.

In [ ]:
oof = []
for seed in SEEDS:
    for fold, (tr, va) in enumerate(group_splits(y, groups, N_SPLITS, seed)):
        Xtr, ytr, gtr = X.iloc[tr].reset_index(drop=True), y[tr], groups[tr]
        p = np.zeros(len(ytr))
        for itr, iva in group_splits(ytr, gtr, N_SPLITS, seed + 1000 + fold):
            p[iva] = fit_pred(make_models(seed)["Extra Trees"], Xtr.iloc[itr], ytr[itr], Xtr.iloc[iva])
        oof.append(dict(seed=seed, fold=fold, n=len(ytr), zeros=int((p == 0).sum()), ones=int((p == 1).sum())))
oof = pd.DataFrame(oof); oof.to_csv(f"{OUT}/inner_oof_counts.csv", index=False)
def summ(sub, label):
    n, z, o = sub.n.sum(), sub.zeros.sum(), sub.ones.sum()
    print(f"{label}: {n:,} inner OOF probabilities | exact 0: {z} ({100*z/n:.2f}%) | exact 1: {o} ({100*o/n:.2f}%)")
summ(oof[oof.seed.isin([0, 1, 2])], "seeds 0-2 (the old three-seed stack run; the paper says 11,868, 0.40% and 0.10%)")
summ(oof, "all seeds used here")

## 8. Text to paste into the paper, and ZIP

In [ ]:
f = lambda v: f"{v:.3f}"
lines = []
for m in MODEL_NAMES:
    lo, hi = ci.loc[m]
    lines.append(f"{m} & {f(t4.loc[m,'Random split'])} & {f(t4.loc[m,'Group'])} & {f(t4.loc[m,'Group + recoded'])} & "
                 f"${t4.loc[m,'Delta']:+.3f}$ & [{f(lo)}, {f(hi)}] & {f(t4.loc[m,'Group PR-AUC'])}\\\\")
tex = "\n".join(lines)
pair_txt = "; ".join(f"{r.a} minus {r.b}: {r['diff']:+.3f} [{r.lo:.3f}, {r.hi:.3f}]" for _, r in pairs.iterrows())
open(f"{OUT}/table4_latex_rows.tex", "w").write(tex + "\n% paired differences: " + pair_txt + "\n")
t4.round(4).to_csv(f"{OUT}/table4_summary.csv"); ci.round(4).to_csv(f"{OUT}/table4_bootstrap_ci.csv"); pairs.round(4).to_csv(f"{OUT}/table4_paired_differences.csv", index=False)
pd.DataFrame({"setting": ["FAST","N_TREES","SEEDS","N_BOOT","N_PERM","n_rows","n_groups","sklearn"],
              "value": [FAST, N_TREES, str(SEEDS), N_BOOT, N_PERM, len(y), len(np.unique(groups)), __import__("sklearn").__version__]}
            ).to_csv(f"{OUT}/run_settings.csv", index=False)
print(tex); print(); print("Paired differences:", pair_txt)
zp = shutil.make_archive("table4_outputs", "zip", root_dir=OUT); print("Created", zp)
try:
    from google.colab import files as cf; cf.download(zp)
except ImportError:
    print("Not in Colab: ZIP is in the working folder.")